In [20]:
!pip install -q transformers datasets peft accelerate sentencepiece rouge-score nltk

In [21]:

import torch
import torch.nn as nn
import numpy as np
from datasets import load_dataset
from transformers import (
    BartTokenizer,
    Trainer,
    TrainingArguments
)
from transformers.models.bart.modeling_bart import BartForConditionalGeneration
from peft import LoraConfig, get_peft_model
from torch.utils.data import Dataset

from rouge_score import rouge_scorer
import nltk
nltk.download("punkt")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [22]:
dataset = load_dataset("raeidsaqur/NIFTY")

train_data = dataset["train"]
val_data   = dataset["valid"]
test_data  = dataset["test"]

tokenizer = BartTokenizer.from_pretrained("facebook/bart-base")


In [23]:
class NIFTYDCCSDataset(Dataset):
    def __init__(self, data, tokenizer, max_len=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        input_text = (
            f"News:\n{item['news']}\n\n"
            f"Market Context:\n{item['context']}\n\n"
            f"Summarize the market movement."
        )

        target_text = (
            item["conversations"][1]["value"]
            if "conversations" in item else
            "Market summary unavailable."
        )

        inputs = self.tokenizer(
            input_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        targets = self.tokenizer(
            target_text,
            truncation=True,
            padding="max_length",
            max_length=128,
            return_tensors="pt"
        )

        labels = targets.input_ids.squeeze()
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": inputs.input_ids.squeeze(),
            "attention_mask": inputs.attention_mask.squeeze(),
            "labels": labels
        }


In [24]:
class DCCSBART(BartForConditionalGeneration):
    def __init__(self, config):
        super().__init__(config)
        hidden = config.d_model
        self.causal_gate = nn.Linear(hidden * 2, 1)
        self.last_alpha = None
    @classmethod
    def _can_set_experts_implementation(cls):
      return False
    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        decoder_input_ids=None,
        decoder_attention_mask=None,
        labels=None,
        encoder_outputs=None,
        **kwargs
    ):
        if encoder_outputs is None:
            encoder_outputs = self.model.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                return_dict=True
            )

            h = encoder_outputs.last_hidden_state
            B, T, H = h.size()
            mid = T // 2

            h_news = h[:, :mid, :]
            h_market = h[:, mid:, :]

            news_repr = h_news.mean(dim=1)
            market_repr = h_market.mean(dim=1)

            alpha = torch.sigmoid(
                self.causal_gate(torch.cat([news_repr, market_repr], dim=-1))
            ).view(B, 1, 1)

            self.last_alpha = alpha.detach().cpu()

            fused_h = torch.cat(
                [alpha * h_news, (1 - alpha) * h_market],
                dim=1
            )

            encoder_outputs.last_hidden_state = fused_h

        return super().forward(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            decoder_attention_mask=decoder_attention_mask,
            labels=labels,
            return_dict=True
        )

    def prepare_inputs_for_generation(
        self,
        decoder_input_ids,
        past_key_values=None,
        attention_mask=None,
        encoder_outputs=None,
        **kwargs
    ):
        return {
            "decoder_input_ids": decoder_input_ids,
            "past_key_values": past_key_values,
            "attention_mask": attention_mask,
            "encoder_outputs": encoder_outputs,
        }


In [25]:
model = DCCSBART.from_pretrained("facebook/bart-base")

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, lora_config)
model = model.to(device)


Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

DCCSBART LOAD REPORT from: facebook/bart-base
Key                | Status  | 
-------------------+---------+-
causal_gate.bias   | MISSING | 
causal_gate.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [26]:
train_dataset = NIFTYDCCSDataset(train_data, tokenizer)
val_dataset   = NIFTYDCCSDataset(val_data, tokenizer)

training_args = TrainingArguments(
    output_dir="./dccs_nifty",
    eval_strategy="steps",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=100,
    eval_steps=100,
    save_steps=500,
    save_total_limit=2,
    num_train_epochs=20,
    report_to="none",
    remove_unused_columns=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer
)


In [ ]:
trainer.train()

trainer.save_model("./dccs_nifty_final")
tokenizer.save_pretrained("./dccs_nifty_final")


Step,Training Loss,Validation Loss
100,14.390994,1.035928


Step,Training Loss,Validation Loss
100,14.390994,1.035928
200,4.212780,0.763468
300,3.518012,0.704595
400,3.315046,0.708208


In [ ]:
model.eval()

sample = test_data[0]

prompt = (
    f"News:\n{sample['news']}\n\n"
    f"Market Context:\n{sample['context']}\n\n"
    f"Summarize the market movement."
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True,
    max_length=512
).to(device)

with torch.no_grad():
    summary_ids = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        num_beams=4
    )

summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print("Generated Summary:\n", summary)
print("α-gate value:", model.last_alpha.item())


In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rougeL"], use_stemmer=True)

rouge1_scores = []
rougeL_scores = []

for i in range(50):
    sample = test_data[i]

    prompt = (
        f"News:\n{sample['news']}\n\n"
        f"Market Context:\n{sample['context']}\n\n"
        f"Summarize the market movement."
    )

    reference = (
        sample["conversations"][1]["value"]
        if "conversations" in sample else ""
    )

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True).to(device)

    with torch.no_grad():
        pred_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128
        )

    prediction = tokenizer.decode(pred_ids[0], skip_special_tokens=True)

    scores = scorer.score(reference, prediction)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

print("ROUGE-1:", np.mean(rouge1_scores))
print("ROUGE-L:", np.mean(rougeL_scores))


In [ ]:
def true_direction_from_pct(pct):
    """
    Ground-truth direction from percentage change.
    Thresholds are conservative to avoid noise.
    """
    if pct > 0.1:
        return "UP"
    elif pct < -0.1:
        return "DOWN"
    else:
        return "NEUTRAL"


def predicted_direction_from_text(text):
    """
    Direction inferred from generated summary.
    """
    text = text.lower()

    if any(w in text for w in [
        "rise", "rising", "rose", "gain", "gained", "up",
        "bullish", "increase", "positive"
    ]):
        return "UP"

    if any(w in text for w in [
        "fall", "fell", "drop", "down", "bearish",
        "decline", "decrease", "negative"
    ]):
        return "DOWN"

    return "NEUTRAL"


In [ ]:
model.eval()

correct = 0
total = 0

N = 50  # evaluate on first 50 test samples

for i in range(N):
    sample = test_data[i]

    prompt = (
        f"News:\n{sample['news']}\n\n"
        f"Market Context:\n{sample['context']}\n\n"
        f"Summarize the market movement."
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_length=128,
            num_beams=4
        )

    summary = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    pred_dir = predicted_direction_from_text(summary)
    true_dir = true_direction_from_pct(sample["pct_change"])

    if pred_dir == true_dir:
        correct += 1

    total += 1

print(f"Directional Faithfulness: {correct / total:.3f}")


In [ ]:
import shutil
import os

# Define the input folder path and output zip file path
# Replace 'path/to/your/folder' with the actual path of the folder you want to zip
input_folder_path = "./dccs_nifty_final"

# Replace 'path/to/your/output.zip' with the desired path and name for the output zip file
# The output zip file will be created in the directory specified by this path
output_zip_path = "./dccs_nifty_final_zipped"

try:
    # Create a zip archive
    # The first argument is the base_name for the archive (e.g., 'dccs_nifty_final_zipped' will create 'dccs_nifty_final_zipped.zip')
    # The second argument is the format, 'zip' for a zip file
    # The third argument is the root_dir, which is the directory to start archiving from
    # The last argument is the base_dir, which is the directory from which to archive the contents
    shutil.make_archive(output_zip_path, 'zip', os.path.dirname(input_folder_path), os.path.basename(input_folder_path))
    print(f"Successfully zipped '{input_folder_path}' to '{output_zip_path}.zip'")
except Exception as e:
    print(f"Error zipping folder: {e}")
